# Scorecard & Findings Builder
Runs the scorecard builder independently.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
from scripts.computation.scorecard_builder import build_from_file

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'
result = build_from_file(phase1_path)

# Save to notebook-local data
(NB_DATA / 'scorecard.json').write_text(
    json.dumps(result, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print('=== Scorecard Dimensions ===')
for d in result['scorecard']['dimensions']:
    print(f'  {d["dimension"]:35s} {d["value"]:.2f}')

print(f'\n=== Findings ({len(result["findings"])}) ===')
for f in result['findings']:
    print(f'  [{f["severity"]:8s}] {f["text"][:80]}')

In [ ]:
raw = json.loads((ROOT / 'data' / 'input' / 'aggregated_scorecard_output.json').read_text(encoding='utf-8'))

print('=== Validation: Scorecard vs Input ===')
all_ok = True

# Validate action correctness matches input mean
raw_ac = sum(c['numeric_metrics']['action_correctness']['mean'] for c in raw['fault_category_scorecards']) / len(raw['fault_category_scorecards'])
sc_dims = {d['dimension']: d['value'] for d in result['scorecard']['dimensions']}
for dim_name, dim_val in sc_dims.items():
    ok = 'PASS' if 0 <= dim_val <= 1 else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} {dim_name}: {dim_val:.2f} in [0,1]')

# Validate finding severities are valid
valid_severities = {'good', 'concern', 'critical', 'note', 'info'}
for f in result['findings']:
    ok = 'PASS' if f['severity'] in valid_severities else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} finding severity: {f["severity"]}')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')